# VenoMap — Reentrenamiento del modelo de IA

Notebook para **Google Colab**. Descarga fotos reales de **iNaturalist**, entrena con
*transfer learning* (MobileNetV2) y exporta `model_unquant.tflite` + `labels.txt`
listos para tu app, **sin tocar el código Kotlin**.

### Antes de empezar
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.
2. Ejecuta las celdas **en orden** (Shift+Enter).
3. La descarga es lo más lento (respeta el límite de la API de iNaturalist).


## 1. Configuración

In [ ]:
# Ajusta estos valores si quieres
IMAGENES_POR_ESPECIE = 150   # sube a 300 para más calidad (tarda más)
TAM_IMG = 224                # entrada del modelo (NO cambiar: la app espera 224)
EPOCAS_BASE = 12             # épocas con la base congelada
EPOCAS_FINETUNE = 8          # épocas afinando las últimas capas
LOTE = 32

import os, time, requests
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))


## 2. Lista de especies
Mismo orden alfabético que tu `labels.txt`, **sin** la clase "Otros"
(esa la sigue dando el umbral de confianza de la app).

In [ ]:
ESPECIES = [
    "Agkistrodon_piscivorus", "Androctonus_australis", "Asthenosoma_varium", "Atrax_robustus",
    "Berberomeloe_majalis", "Bitis_gabonica", "Bungarus_caeruleus", "Calloselasma_rhodostoma",
    "Carukia_barnesi", "Centruroides_sculpturatus", "Chironex_fleckeri", "Conus_geographus",
    "Crotalus_adamanteus", "Daboia_russelii", "Dasyatis_pastinaca", "Dendroaspis_polylepis",
    "Dendrobates_tinctorius", "Enhydrina_schistosa", "Hadronyche_formidabilis", "Hapalochlaena_maculosa",
    "Heloderma_suspectum", "Latrodectus_mactans", "Leiurus_quinquestriatus", "Lonomia_obliqua",
    "Loxosceles_laeta", "Micrurus_fulvius", "Naja_naja", "Naja_nigricollis",
    "Notechis_scutatus", "Nycticebus_coucang", "Ophiophagus_hannah", "Ornithorhynchus_anatinus",
    "Oxyuranus_microlepidotus", "Paraponera_clavata", "Phoneutria_nigriventer", "Phyllobates_terribilis",
    "Phyllomedusa_sauvagii", "Physalia_physalis", "Pitohui_dichrous", "Pterois_volitans",
    "Salamandra_salamandra", "Scolopendra_cingulata", "Scolopendra_gigantea", "Steatoda_nobilis",
    "Synanceia_horrida", "Takifugu_rubripes", "Varanus_komodoensis", "Vespa_mandarinia",
    "Vipera_latastei",
]
print(len(ESPECIES), 'especies')


## 3. Descargar imágenes de iNaturalist
Busca cada especie por su nombre científico y baja las fotos mejor valoradas
de observaciones *research grade*.

In [ ]:
DIR_DATOS = '/content/dataset'
os.makedirs(DIR_DATOS, exist_ok=True)

def taxon_id(nombre_cientifico):
    r = requests.get('https://api.inaturalist.org/v1/taxa',
                     params={'q': nombre_cientifico, 'rank': 'species', 'per_page': 1})
    res = r.json().get('results', [])
    return res[0]['id'] if res else None

def descargar_especie(nombre, objetivo):
    carpeta = os.path.join(DIR_DATOS, nombre)
    os.makedirs(carpeta, exist_ok=True)
    descargadas = len(os.listdir(carpeta))
    if descargadas >= objetivo:
        print(f'  {nombre}: ya hay {descargadas}, salto'); return
    tid = taxon_id(nombre.replace('_', ' '))
    if not tid:
        print(f'  {nombre}: NO encontrado en iNaturalist'); return
    pagina = 1
    while descargadas < objetivo and pagina <= 10:
        r = requests.get('https://api.inaturalist.org/v1/observations',
            params={'taxon_id': tid, 'photos': 'true', 'quality_grade': 'research',
                    'per_page': 200, 'page': pagina, 'order_by': 'votes'})
        obs = r.json().get('results', [])
        if not obs: break
        for o in obs:
            for p in o.get('photos', []):
                if descargadas >= objetivo: break
                url = p['url'].replace('square', 'medium')
                try:
                    img = requests.get(url, timeout=15).content
                    with open(os.path.join(carpeta, f'{descargadas}.jpg'), 'wb') as f:
                        f.write(img)
                    descargadas += 1
                except Exception:
                    pass
        pagina += 1
        time.sleep(1)  # respetar el limite de la API (1 req/seg)
    print(f'  {nombre}: {descargadas} imagenes')

for i, esp in enumerate(ESPECIES):
    print(f'[{i+1}/{len(ESPECIES)}] {esp}')
    descargar_especie(esp, IMAGENES_POR_ESPECIE)
print('Descarga terminada')


## 4. Limpieza y recuento
Borra imágenes corruptas y enseña cuántas hay por especie.

In [ ]:
import PIL.Image
total = 0
for esp in sorted(os.listdir(DIR_DATOS)):
    carpeta = os.path.join(DIR_DATOS, esp)
    buenas = 0
    for fn in list(os.listdir(carpeta)):
        ruta = os.path.join(carpeta, fn)
        try:
            PIL.Image.open(ruta).verify(); buenas += 1
        except Exception:
            os.remove(ruta)
    total += buenas
    aviso = '  <-- POCAS' if buenas < 40 else ''
    print(f'{esp}: {buenas}{aviso}')
print('TOTAL imagenes:', total)


## 5. Preparar los datasets (train / validación + aumento)

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_ds = tf.keras.utils.image_dataset_from_directory(
    DIR_DATOS, validation_split=0.2, subset='training', seed=123,
    image_size=(TAM_IMG, TAM_IMG), batch_size=LOTE, label_mode='categorical')
val_ds = tf.keras.utils.image_dataset_from_directory(
    DIR_DATOS, validation_split=0.2, subset='validation', seed=123,
    image_size=(TAM_IMG, TAM_IMG), batch_size=LOTE, label_mode='categorical')

CLASES = train_ds.class_names
print(len(CLASES), 'clases')

aumento = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE
def preparar(ds, entrenamiento=False):
    if entrenamiento:
        ds = ds.map(lambda x, y: (aumento(x), y), num_parallel_calls=AUTOTUNE)
    # preprocess_input normaliza a [-1, 1] = lo mismo que hace la app
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_p = preparar(train_ds, True)
val_p = preparar(val_ds, False)


## 6. Entrenar (base congelada)

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(TAM_IMG, TAM_IMG, 3),
                                         include_top=False, weights='imagenet')
base.trainable = False

modelo = tf.keras.Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(CLASES), activation='softmax'),
])
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='categorical_crossentropy', metrics=['accuracy'])
modelo.fit(train_p, validation_data=val_p, epochs=EPOCAS_BASE)


## 7. Afinar (fine-tuning de las últimas capas)

In [ ]:
base.trainable = True
for capa in base.layers[:-30]:
    capa.trainable = False
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
               loss='categorical_crossentropy', metrics=['accuracy'])
modelo.fit(train_p, validation_data=val_p, epochs=EPOCAS_FINETUNE)


## 8. Exportar a TFLite + labels.txt
El modelo espera entrada normalizada a `[-1, 1]`, exactamente lo que hace tu app.

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
tflite = conv.convert()
with open('model_unquant.tflite', 'wb') as f:
    f.write(tflite)
with open('labels.txt', 'w') as f:
    f.write('\n'.join(CLASES))
print('Generados model_unquant.tflite y labels.txt')
print('Orden de clases:', CLASES)


## 9. Descargar los archivos

In [ ]:
from google.colab import files
files.download('model_unquant.tflite')
files.download('labels.txt')


## Cómo meterlo en la app
1. Sustituye `AppVenenos/app/src/main/assets/model_unquant.tflite` por el nuevo.
2. Sustituye `AppVenenos/app/src/main/assets/labels.txt` por el nuevo (49 líneas).
3. Recompila la APK. **No hay que tocar código.**

La clase "Otros" la da el umbral de confianza de la app (si la mejor < 70%).
